### LeNet-5 Architecture Explanation

LeNet-5 is one of the earliest and most influential convolutional neural networks (CNNs), developed by Yann LeCun et al. in 1998 for handwritten digit recognition (specifically, MNIST). It laid the groundwork for modern CNNs with its use of convolutional layers, pooling layers, and fully connected layers.

Here's a breakdown of its typical architecture:

1.  **Input Layer**: Accepts a grayscale image (e.g., 32x32 pixels, padded from 28x28 for MNIST).

2.  **C1 (Convolutional Layer)**:
    *   Applies 6 convolutional filters (kernels) of size 5x5.
    *   Strides: 1x1.
    *   Activation: Typically `tanh` (though `ReLU` is common in modern CNNs).
    *   Output: 6 feature maps (28x28 each).

3.  **S2 (Subsampling/Pooling Layer)**:
    *   Performs average pooling over 2x2 windows.
    *   Strides: 2x2.
    *   Output: 6 downsampled feature maps (14x14 each).

4.  **C3 (Convolutional Layer)**:
    *   Applies 16 convolutional filters of size 5x5.
    *   **Sparse Connections**: Historically, C3 was sparsely connected to S2 to reduce parameters. Modern implementations often use full connectivity for simplicity and performance with better hardware.
    *   Strides: 1x1.
    *   Activation: `tanh`.
    *   Output: 16 feature maps (10x10 each).

5.  **S4 (Subsampling/Pooling Layer)**:
    *   Performs average pooling over 2x2 windows.
    *   Strides: 2x2.
    *   Output: 16 downsampled feature maps (5x5 each).

6.  **C5 (Convolutional Layer)**:
    *   This is often treated as a fully connected layer because the 5x5 input maps are flattened.
    *   Applies 120 filters (each 5x5, resulting in 1x1 output per filter).
    *   Output: 120 feature maps (1x1 each), effectively 120 neurons.
    *   Activation: `tanh`.

7.  **F6 (Fully Connected Layer)**:
    *   Connects the 120 neurons from C5 to 84 neurons.
    *   Activation: `tanh`.

8.  **Output Layer**:
    *   A fully connected layer with 10 neurons (one for each digit 0-9).
    *   Activation: `softmax` for classification, yielding probability distribution over classes.

LeNet-5 demonstrated the power of hierarchical feature extraction using convolutions and pooling, which are fundamental concepts in most deep learning vision models today.

### Kaggle API Key Setup

To use your Kaggle API key, you first need to generate one from your Kaggle account settings (under the 'API' section, click 'Create New API Token'). This will download a `kaggle.json` file.

Then, in Colab, you can upload this `kaggle.json` content to your Colab Secrets (the key icon in the left panel). Name the secret `kaggle1` as you've indicated. The content of the secret should be the entire JSON string from your `kaggle.json` file (e.g., `{"username":"YOUR_USERNAME","key":"YOUR_API_KEY"}`).

Once set, the following code will configure Kaggle to use your credentials:

In [1]:
import os
import json
from google.colab import userdata

# Retrieve the Kaggle API key from Colab secrets
kaggle_credentials_json = userdata.get('kaggle1')

# Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Write the kaggle.json file
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write(kaggle_credentials_json)

# Set file permissions for security
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API key configured successfully!")

Kaggle API key configured successfully!


### Downloading a Kaggle Dataset

Now you can download datasets from Kaggle. Replace `'YOUR_DATASET_NAME'` with the actual dataset path (e.g., `titanic` for the Titanic competition, or `zalando-research/fashion-mnist` for a dataset).

If it's a competition, use `kaggle competitions download -c COMPETITION_NAME`.
If it's a dataset, use `kaggle datasets download -d DATASET_OWNER/DATASET_NAME`.

In [2]:
# Example: Download a competition dataset (uncomment and replace with your desired dataset)
# !kaggle competitions download -c digit-recognizer

# Example: Download a general dataset (uncomment and replace with your desired dataset)
# !kaggle datasets download -d zalando-research/fashion-mnist

# You might need to unzip the downloaded files
# import zipfile
# with zipfile.ZipFile('downloaded_file.zip', 'r') as zip_ref:
#     zip_ref.extractall('extracted_folder')

### LeNet-5 Implementation with MNIST

Here's a Keras implementation of LeNet-5 for the MNIST handwritten digit classification task. MNIST is a classic dataset for demonstrating LeNet-5 due to its simplicity and suitability for the original architecture.

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# 1. Load and preprocess the MNIST dataset
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Reshape images to (height, width, channels) for CNN input
# For LeNet-5, original input is 32x32, so we'll pad 28x28 to 32x32
# Keras Conv2D handles padding, but traditionally LeNet-5 used explicit padding.
# We'll use 28x28 directly and let Keras handle internal padding if needed or adjust first conv output

train_images = train_images.reshape((60000, 28, 28, 1)).astype('float32') / 255
test_images = test_images.reshape((10000, 28, 28, 1)).astype('float32') / 255

# Convert labels to one-hot encoding
train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

print("MNIST data loaded and preprocessed.")
print(f"Train images shape: {train_images.shape}")
print(f"Test images shape: {test_images.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
MNIST data loaded and preprocessed.
Train images shape: (60000, 28, 28, 1)
Test images shape: (10000, 28, 28, 1)


In [4]:
# 2. Define the LeNet-5 model architecture
def create_lenet5_model(input_shape=(28, 28, 1), num_classes=10):
    model = models.Sequential()

    # C1: Convolutional Layer
    # Input: 28x28x1. With 5x5 kernel, 6 filters, 'same' padding, output is 28x28x6
    # Original LeNet-5 used 32x32 input and 'valid' padding to get 28x28 output
    # Here we'll start with 28x28 and target similar output dimensions
    model.add(layers.Conv2D(6, kernel_size=(5, 5), activation='tanh', input_shape=input_shape, padding='same')) # Output: 28x28x6

    # S2: Average Pooling Layer
    model.add(layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))) # Output: 14x14x6

    # C3: Convolutional Layer
    # Output: 10x10x16 (if padding='valid') or 14x14x16 (if padding='same')
    # Using 'valid' to mimic original LeNet-5 behavior more closely for feature map reduction
    model.add(layers.Conv2D(16, kernel_size=(5, 5), activation='tanh', padding='valid')) # Output: 10x10x16

    # S4: Average Pooling Layer
    model.add(layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))) # Output: 5x5x16

    # Flatten for fully connected layers
    model.add(layers.Flatten()) # Output: 5*5*16 = 400

    # C5: Fully Connected Layer (historically a Conv layer treated as FC)
    model.add(layers.Dense(120, activation='tanh'))

    # F6: Fully Connected Layer
    model.add(layers.Dense(84, activation='tanh'))

    # Output Layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

# Create the model instance
lenet5_model = create_lenet5_model()

# Display the model summary
lenet5_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# 3. Compile the model
lenet5_model.compile(optimizer='adam',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

print("Model compiled.")

Model compiled.


In [6]:
# 4. Train the model
history = lenet5_model.fit(train_images, train_labels, epochs=10, batch_size=64, validation_split=0.1)

print("Model training complete.")

Epoch 1/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.8517 - loss: 0.5077 - val_accuracy: 0.9683 - val_loss: 0.1113
Epoch 2/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 40s 38ms/step - accuracy: 0.9630 - loss: 0.1188 - val_accuracy: 0.9798 - val_loss: 0.0730
Epoch 3/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 32s 38ms/step - accuracy: 0.9762 - loss: 0.0767 - val_accuracy: 0.9802 - val_loss: 0.0709
Epoch 4/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 32s 38ms/step - accuracy: 0.9819 - loss: 0.0597 - val_accuracy: 0.9823 - val_loss: 0.0612
Epoch 5/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 31s 37ms/step - accuracy: 0.9872 - loss: 0.0424 - val_accuracy: 0.9807 - val_loss: 0.0645
Epoch 6/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 41s 37ms/step - accuracy: 0.9881 - loss: 0.0368 - val_accuracy: 0.9828 - val_loss: 0.0559
Epoch 7/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 33s 39ms/step - accuracy: 0.9903 - loss: 0.0314 - val_accuracy: 0.9860 - val_loss: 0.0507
Epoch 8/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 31s 37ms/step - accuracy: 0.9922 - loss: 0.0238 - 

In [7]:
# 5. Evaluate the model
test_loss, test_acc = lenet5_model.evaluate(test_images, test_labels)

print(f"\nTest accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9803 - loss: 0.0627

Test accuracy: 0.9832
Test loss: 0.0553
